# Assignment 6: Mining and Proof-of-Work Lab

**BLCH9X2 Blockchain, Master of Financial Engineering, University of Johannesburg** Bukho Ndziweni, student number 220081696, 15 September 2026

Implements a proof-of-work miner searching for a nonce against a numeric difficulty target, measures attempts and wall-clock time across several difficulty levels, credits a block reward to a miner address, and implements simplified difficulty retargeting. All timing figures are measured on the machine that ran this notebook.

## 1. Setup: the difficulty target, and the arithmetic behind it

A SHA-256 hash is 64 hexadecimal characters, which is a very large whole number between 0 and 2²⁵⁶ − 1. Mining means searching for a nonce whose hash, read as that number, falls at or below an agreed ceiling called the target. A low target is hard because few numbers fall below it.

**The rule this lab uses, as required by Part A (a):** a block is mined when its hash, read as a number, is at or below `MAX_TARGET` divided by the difficulty. This is the numeric target form rather than a count of leading zeros.

Two reasons for that choice. Part B has to multiply the target by fractional amounts during retargeting, which a count of zeros cannot express, since zeros only move in steps of sixteen. And the expected number of attempts falls out exactly as the difficulty itself, which gives the parameter study in Part A (b) a theoretical column to be measured against.

In [1]:
# ============================================================================
# SETUP: THE DIFFICULTY TARGET, AND THE ARITHMETIC BEHIND IT
# ============================================================================

import hashlib          # hashlib gives us SHA-256 (Secure Hash Algorithm 256-bit), the fingerprint function
import json             # json turns Python dictionaries into text in a controlled, repeatable way
import time             # time lets us measure wall-clock seconds, which the parameter study needs
import statistics       # statistics gives us mean and median for summarising repeated mining trials

# ---------------------------------------------------------------------------
# How a difficulty target works
# ---------------------------------------------------------------------------
# A SHA-256 hash is 64 hexadecimal characters, which is really just a very large
# whole number between 0 and 2^256 - 1. Mining means searching for a nonce whose
# hash, read as that number, lands BELOW an agreed ceiling called the target.
#
# A low target is hard, because few numbers fall below it. A high target is easy.
# Difficulty is simply how many times smaller the target is than the maximum:
#
#       target = MAX_TARGET / difficulty
#
# This is the numeric target form. The alternative, counting leading zeros, is the
# same idea in cruder steps: "four leading zeros" is exactly the target 0000ffff...
# The numeric form is used here because Part B has to multiply the target by
# fractional amounts, which a count of zeros cannot express.

MAX_TARGET = 2 ** 256 - 1        # the largest value a SHA-256 hash can take, which is difficulty 1 where every hash wins

def target_for_difficulty(difficulty: int) -> int:
    """Convert a difficulty number into the ceiling a winning hash must fall below."""
    return MAX_TARGET // difficulty        # integer division, so a difficulty of 256 gives a target 256 times smaller

def hash_to_number(hash_hex: str) -> int:
    """Read a 64 character hash as the very large whole number it actually is."""
    return int(hash_hex, 16)               # base 16 because the hash is written in hexadecimal

def meets_target(hash_hex: str, target: int) -> bool:
    """Return True when this hash counts as a winning one, meaning it falls at or below the target."""
    return hash_to_number(hash_hex) <= target   # the single test that decides whether mining stops

def sha256_hex(text: str) -> str:
    """Return the SHA-256 fingerprint of a piece of text, written as 64 hexadecimal characters."""
    encoded_text = text.encode("utf-8")    # convert the text into raw bytes, because hashing works on bytes
    return hashlib.sha256(encoded_text).hexdigest()   # hash those bytes and return readable hexadecimal

def canonical_json(payload: dict) -> str:
    """Turn a dictionary into one single agreed text string, so hashing is repeatable."""
    # sort_keys=True   : fields always written in alphabetical order, so field order cannot change the hash
    # separators=(...) : no optional spaces, so spacing cannot change the hash
    return json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=True)

# ===========================================================================
# SELF-TEST: confirm the target arithmetic behaves as claimed
# ===========================================================================
print("TARGET ARITHMETIC CHECK")                                              # heading for this block of output
print("=" * 78)                                                               # heavy divider line
print(f"Maximum possible hash value : {MAX_TARGET:064x}")                     # the ceiling at difficulty 1, printed as 64 hex characters
print()                                                                       # blank line for readability

print(f"{'Difficulty':<14}{'Target (first 20 hex characters)':<36}{'Expected attempts'}")  # table header
print("-" * 78)                                                                            # divider under the header
for difficulty in [1, 16, 256, 65536, 1048576]:                               # five difficulty settings, each 16 times harder than the last
    target = target_for_difficulty(difficulty)                                # the ceiling this difficulty demands
    target_as_hex = f"{target:064x}"                                          # write the target as 64 hexadecimal characters
    print(f"{difficulty:<14,}{target_as_hex[:20]:<36}{difficulty:,}")         # expected attempts equals the difficulty itself
print("-" * 78)                                                               # divider under the table
print()                                                                       # blank line for readability

# Why expected attempts equals the difficulty: a hash is equally likely to land anywhere
# between 0 and MAX_TARGET, so the chance of landing at or below MAX_TARGET / d is 1 in d.
# On average, therefore, d attempts are needed. This is the theoretical column the
# parameter study in Part A (b) is measured against.

print("The numeric target and the leading zeros rule are the same idea")                  # what the next lines establish
difficulty_65536 = target_for_difficulty(65536)                                           # the target at difficulty 65,536
print(f"  Target at difficulty 65,536 : {difficulty_65536:064x}"[:56] + "...")            # its first characters, shown truncated
print(f"  A hash must therefore begin with four zeros to fall below it.")                 # 16^4 = 65,536, which is four hex characters
print(f"  '0000ab...' wins  : {meets_target('0000ab' + '0' * 58, difficulty_65536)} (expected: True)")   # four zeros clears the ceiling
print(f"  '0001ab...' wins  : {meets_target('0001ab' + '0' * 58, difficulty_65536)} (expected: False)")  # only three zeros does not
print()                                                                                   # blank line for readability

print("This lab therefore states its rule as: a block is mined when its hash, read as a")  # the formal statement for the report
print("number, is at or below MAX_TARGET divided by the difficulty.")                      # the numeric target form, as required by (a)

TARGET ARITHMETIC CHECK
Maximum possible hash value : ffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffff

Difficulty    Target (first 20 hex characters)    Expected attempts
------------------------------------------------------------------------------
1             ffffffffffffffffffff                1
16            0fffffffffffffffffff                16
256           00ffffffffffffffffff                256
65,536        0000ffffffffffffffff                65,536
1,048,576     00000fffffffffffffff                1,048,576
------------------------------------------------------------------------------

The numeric target and the leading zeros rule are the same idea
  Target at difficulty 65,536 : 0000ffffffffffffffffffff...
  A hash must therefore begin with four zeros to fall below it.
  '0000ab...' wins  : True (expected: True)
  '0001ab...' wins  : False (expected: False)

This lab therefore states its rule as: a block is mined when its hash, read as a
number, is at or 

## 2. The miner: searching for a nonce that meets the target

Mining is a guessing game with one legal move. Every field of a block is fixed by what the block contains, except the nonce, which exists purely so there is something to change. The miner sets the nonce to 0, hashes the block, checks whether the hash falls at or below the target, and if not adds one and tries again.

There is no cleverer method. SHA-256 gives no hint about which nonce to try next, so the only strategy is exhaustive search, which is precisely why the work costs real time and real electricity.

Timestamps are fixed constants rather than clock readings and the search always starts at nonce zero, so the winning nonce and hash reproduce on any machine. The wall-clock seconds and hash rate do not: those are measured on whichever machine runs the notebook, and they are the figures the parameter study in section 3 reports.

In [2]:
# ============================================================================
# THE MINER: SEARCHING FOR A NONCE THAT MEETS THE TARGET
# ----------------------------------------------------------------------------
# Mining is a guessing game with one legal move. Every field of a block is fixed
# by what the block contains, except the nonce, which exists purely so there is
# something to change. The miner sets the nonce to 0, hashes the block, checks
# whether that hash falls at or below the target, and if not, adds one and tries
# again. There is no cleverer method: SHA-256 gives no hint about which nonce to
# try next, which is exactly why the search costs real time and real electricity.
# ============================================================================

BASE_TIMESTAMP = 1757894400     # a fixed Unix timestamp (15 September 2026, 00:00:00 UTC) used instead of the live clock

def build_header(index: int, previous_hash: str, transactions: list, difficulty: int,
                 timestamp: int = BASE_TIMESTAMP, nonce: int = 0) -> dict:
    """Assemble the fields a miner hashes over, with the nonce left as the field to be searched."""
    return {                                  # a dictionary holding everything the block commits to
        "index": index,                       # the position this block would occupy in a chain
        "previous_hash": previous_hash,       # the hash of the block before it, which is what links a chain together
        "transactions": transactions,         # the payments this block carries, including the miner's reward
        "difficulty": difficulty,             # committing to the difficulty stops a miner claiming easy work was hard
        "timestamp": timestamp,               # when the block was made, fixed here so nonces reproduce on any machine
        "nonce": nonce,                       # the only field the miner is free to change
    }

def header_hash(header: dict) -> str:
    """Fingerprint a block header as it currently stands."""
    return sha256_hex(canonical_json(header))   # one agreed text form (Cell 1), then SHA-256 over it

def mine(header: dict, difficulty: int, report_every: int = 0) -> dict:
    """Search nonce values until the block hash meets the target, and measure the effort that took."""
    target = target_for_difficulty(difficulty)   # convert the difficulty into the ceiling the hash must fall below
    working_header = dict(header)                # copy the header so the caller's dictionary is not modified
    working_header["difficulty"] = difficulty    # make sure the header commits to the difficulty actually being mined
    attempts = 0                                 # counts every hash computed during this search
    nonce = 0                                    # the search always begins at zero, which keeps the winning nonce reproducible

    start_time = time.perf_counter()             # perf_counter is the clock meant for timing intervals, not calendar dates

    while True:                                  # keep guessing until a winning hash appears
        working_header["nonce"] = nonce          # write the current guess into the header
        candidate_hash = header_hash(working_header)   # fingerprint the header with that guess in place
        attempts += 1                            # that was one more hash computed
        if meets_target(candidate_hash, target): # does this hash fall at or below the target?
            break                                # yes, so the block is mined and the search stops
        if report_every and attempts % report_every == 0:        # optional progress line for long searches
            print(f"    ... {attempts:,} attempts so far")        # printed only when the caller asks for it
        nonce += 1                               # no, so try the next nonce

    elapsed_seconds = time.perf_counter() - start_time    # wall-clock time the search actually took on this machine

    return {                                                              # everything the parameter study needs
        "nonce": nonce,                                                   # the winning nonce
        "hash": candidate_hash,                                           # the winning hash
        "attempts": attempts,                                             # how many hashes were computed to find it
        "seconds": elapsed_seconds,                                       # how long that took in wall-clock seconds
        "hash_rate": attempts / elapsed_seconds if elapsed_seconds else 0, # hashes per second on this machine
        "header": working_header,                                         # the finished header, nonce included
    }

# ===========================================================================
# MINE ONE BLOCK AND INSPECT THE RESULT
# ===========================================================================
print("MINING ONE BLOCK")                                                    # heading for this block of output
print("=" * 78)                                                              # heavy divider line

demo_difficulty = 65536                                                      # difficulty 65,536, equivalent to four leading zeros

demo_header = build_header(                                                  # build a header to mine
    index=1,                                                                 # block number 1
    previous_hash="0" * 64,                                                  # 64 zeros, the convention meaning there is no earlier block
    transactions=[{"sender": "BK", "receiver": "Thandi", "amount": 250.00}],  # one payment, for something to commit to
    difficulty=demo_difficulty,                                              # the difficulty this block is being mined at
)

result = mine(demo_header, demo_difficulty)                                  # run the search and measure it

print(f"Difficulty        : {demo_difficulty:,}")                            # the difficulty setting used
print(f"Target ceiling    : {target_for_difficulty(demo_difficulty):064x}"[:38] + "...")  # the ceiling the hash had to fall below
print(f"Winning nonce     : {result['nonce']:,}")                            # the nonce the search landed on
print(f"Winning hash      : {result['hash']}")                               # the hash that nonce produced
print(f"Attempts needed   : {result['attempts']:,}")                         # how many hashes were computed
print(f"Expected attempts : {demo_difficulty:,}")                            # what theory predicts on average
print(f"Wall-clock time   : {result['seconds']:.3f} seconds")                # measured on this machine, so it varies by processor
print(f"Hash rate         : {result['hash_rate']:,.0f} hashes per second")   # the speed this machine achieved
print()                                                                      # blank line for readability

print("Checking the result")                                                              # heading for the verification lines
print(f"  Hash falls at or below the target : {meets_target(result['hash'], target_for_difficulty(demo_difficulty))}")  # must be True
print(f"  Re-hashing the header reproduces it: {header_hash(result['header']) == result['hash']}")  # the work can be checked instantly
print()                                                                                   # blank line for readability

print("Note the asymmetry: that search took hundreds of thousands of hashes, but anyone")  # the point proof of work rests on
print("can confirm it with a single hash. Expensive to produce, cheap to verify.")         # costly to create, trivial to check

MINING ONE BLOCK
Difficulty        : 65,536
Target ceiling    : 0000ffffffffffffff...
Winning nonce     : 104,318
Winning hash      : 000073385136b237f3b2df759ba30b0545260de38d3558cd1aaa4febea9ba918
Attempts needed   : 104,319
Expected attempts : 65,536
Wall-clock time   : 1.804 seconds
Hash rate         : 57,829 hashes per second

Checking the result
  Hash falls at or below the target : True
  Re-hashing the header reproduces it: True

Note the asymmetry: that search took hundreds of thousands of hashes, but anyone
can confirm it with a single hash. Expensive to produce, cheap to verify.


## 3. Parameter study: attempts and wall-clock time across difficulty levels

Part A (b) asks for attempts and wall-clock time at three or more difficulty levels. Five are used here, each four times harder than the one before.

Each level is mined five times rather than once. Mining is a random search, so a single run can land far from the average, as section 2 showed at 104,319 attempts against an expectation of 65,536. Reporting a mean and a median across repeated trials is the difference between a measurement and an anecdote.

Every trial mines a different header, varied by block index. Re-mining identical content would repeat the same nonce and produce five copies of one result rather than five independent samples.

**All timing figures below are measured on the machine that ran this notebook.** Raise `TRIALS_PER_LEVEL` for tighter averages, at the cost of a longer run.

In [3]:
# ============================================================================
# PARAMETER STUDY: ATTEMPTS AND WALL-CLOCK TIME ACROSS DIFFICULTY LEVELS
# ----------------------------------------------------------------------------
# Part A (b) asks for attempts and wall-clock time at three or more difficulty
# levels. Five are used here, each four times harder than the one before.
#
# Each level is mined several times rather than once. Mining is a random search,
# so a single run can land far from the average, as Cell 2 already showed at 104,319
# attempts against an expectation of 65,536. Repeating the trial and reporting the
# mean and the median is the difference between a measurement and an anecdote.
#
# Every trial mines a DIFFERENT header, because re-mining identical content would
# simply repeat the same nonce and produce five copies of one result rather than
# five independent samples. The block index is varied to achieve this.
# ============================================================================

DIFFICULTY_LEVELS = [1024, 4096, 16384, 65536, 262144]   # the difficulties to study, each four times the previous one
TRIALS_PER_LEVEL = 5                                     # how many independent blocks to mine at each difficulty

print("PARAMETER STUDY")                                                         # heading for this block of output
print("=" * 78)                                                                  # heavy divider line
print(f"Difficulty levels : {', '.join(f'{d:,}' for d in DIFFICULTY_LEVELS)}")    # state the settings being measured
print(f"Trials per level  : {TRIALS_PER_LEVEL}")                                 # state how many samples each figure rests on
print("Mining now. Every number below is measured on the machine running this notebook.")  # make the provenance explicit
print()                                                                          # blank line for readability

study_results = []                                       # collects one summary record per difficulty level

for difficulty in DIFFICULTY_LEVELS:                     # work through the difficulty levels from easiest to hardest
    attempts_per_trial = []                              # every trial's attempt count at this difficulty
    seconds_per_trial = []                               # every trial's wall-clock time at this difficulty

    for trial_number in range(1, TRIALS_PER_LEVEL + 1):  # run the trials one after another
        trial_header = build_header(                     # build a header unique to this trial
            index=trial_number,                          # varying the index makes each trial an independent search
            previous_hash="0" * 64,                      # a fixed placeholder parent hash, identical across all trials
            transactions=[{"note": f"parameter study trial {trial_number}"}],  # a marker record so the block carries content
            difficulty=difficulty,                       # the difficulty being measured
        )
        trial_result = mine(trial_header, difficulty)    # mine it and measure the effort
        attempts_per_trial.append(trial_result["attempts"])   # record how many hashes this trial needed
        seconds_per_trial.append(trial_result["seconds"])     # record how long this trial took

    total_attempts = sum(attempts_per_trial)                              # all hashes computed at this difficulty
    total_seconds = sum(seconds_per_trial)                                # all time spent at this difficulty
    study_results.append({                                                # store the summary for the tables below
        "difficulty": difficulty,                                         # the difficulty this record describes
        "attempts_list": attempts_per_trial,                              # the individual trial results, kept for the detail table
        "mean_attempts": statistics.mean(attempts_per_trial),             # average hashes needed across the trials
        "median_attempts": statistics.median(attempts_per_trial),         # the middle value, less swayed by one unlucky run
        "mean_seconds": statistics.mean(seconds_per_trial),               # average wall-clock time per block
        "total_attempts": total_attempts,                                 # total hashes at this difficulty
        "total_seconds": total_seconds,                                   # total time at this difficulty
        "hash_rate": total_attempts / total_seconds,                      # measured speed, hashes per second
    })
    print(f"  Difficulty {difficulty:>9,} done: "                          # progress line so the wait is not silent
          f"{total_attempts:>10,} hashes in {total_seconds:6.2f} seconds")  # what this level cost in total

print()                                                                   # blank line for readability

# ---------------------------------------------------------------------------
# Detail table: every individual trial
# ---------------------------------------------------------------------------
print("ATTEMPTS PER TRIAL")                                                      # heading for the detail table
print("-" * 78)                                                                  # divider line
trial_columns = "".join(f"Trial {n:<7}" for n in range(1, TRIALS_PER_LEVEL + 1))  # build the trial column headings
print(f"{'Difficulty':<13}{trial_columns}")                                       # table header
print("-" * 78)                                                                   # divider under the header
for record in study_results:                                                      # one row per difficulty level
    trial_cells = "".join(f"{a:<13,}" for a in record["attempts_list"])            # each trial's attempt count
    print(f"{record['difficulty']:<13,}{trial_cells}")                             # print the row
print("-" * 78)                                                                    # divider under the table
print("The spread within a single row is the point: mining is a random search, so")  # explain why the spread matters
print("individual runs scatter widely around the average.")                          # and why one run proves little
print()                                                                              # blank line for readability

# ---------------------------------------------------------------------------
# Summary table, formatted for pasting straight into the report
# ---------------------------------------------------------------------------
print("TIMING TABLE FOR THE REPORT")                                                       # heading for the main deliverable
print("=" * 78)                                                                            # heavy divider line
print(f"{'Difficulty':<12}{'Expected':<12}{'Mean':<12}{'Median':<12}{'Mean secs':<12}{'Hashes/sec'}")  # table header
print("-" * 78)                                                                            # divider under the header
for record in study_results:                                                               # one row per difficulty level
    print(f"{record['difficulty']:<12,}"                                                   # the difficulty setting
          f"{record['difficulty']:<12,}"                                                   # expected attempts, which equals the difficulty
          f"{record['mean_attempts']:<12,.0f}"                                             # measured mean attempts
          f"{record['median_attempts']:<12,.0f}"                                           # measured median attempts
          f"{record['mean_seconds']:<12.3f}"                                               # measured mean wall-clock seconds per block
          f"{record['hash_rate']:,.0f}")                                                   # measured hash rate at this difficulty
print("-" * 78)                                                                            # divider under the table

overall_attempts = sum(r["total_attempts"] for r in study_results)   # every hash computed across the whole study
overall_seconds = sum(r["total_seconds"] for r in study_results)     # every second spent across the whole study

print(f"Whole study: {overall_attempts:,} hashes in {overall_seconds:.2f} seconds, "  # the headline totals
      f"{overall_attempts / overall_seconds:,.0f} hashes per second overall")         # and the overall measured rate
print()                                                                               # blank line for readability

# ---------------------------------------------------------------------------
# Reading the result
# ---------------------------------------------------------------------------
slowest_rate = min(r["hash_rate"] for r in study_results)      # the lowest hash rate measured across the levels
fastest_rate = max(r["hash_rate"] for r in study_results)      # the highest hash rate measured across the levels
rate_spread = (fastest_rate / slowest_rate - 1) * 100          # how much the two differ, as a percentage

print("WHAT THE TABLE SHOWS")                                                            # heading for the interpretation
print("  1. Expected attempts equals the difficulty exactly, because a hash lands at or") # the theoretical column
print("     below MAX_TARGET / d with probability 1 in d.")                               # and why that is so
print("  2. Mean attempts track that expectation, loosely at five trials and more closely")  # what the measurement shows
print("     as trials increase. Five is few: the mean still moves noticeably between runs.")  # an honest caveat
print("  3. Mean time per block rises in step with difficulty, four times longer for each") # the wall-clock consequence
print("     four times harder setting, because the hash rate is a property of the machine") # the reason the two track
print("     rather than of the difficulty.")                                                # difficulty does not change speed
print(f"  4. The measured hash rate ranges from {slowest_rate:,.0f} to {fastest_rate:,.0f} per second,")  # the measured spread
print(f"     a difference of {rate_spread:.0f} percent across the study.")                  # stated rather than assumed
if rate_spread < 15:                                                                        # a narrow spread needs no explanation
    print("     That is narrow enough to treat the rate as constant, which confirms that")  # what a flat column means
    print("     difficulty alone changed between the levels.")                              # the internal consistency check
else:                                                                                       # a wide spread is itself a finding
    print("     That is wide, and the rate rises with difficulty rather than varying at")   # describe the pattern
    print("     random, which points at processor frequency scaling: the easiest levels")   # the most likely cause
    print("     finish in a fraction of a second, before the processor has ramped up from") # why short runs measure low
    print("     its idle clock speed, while the hardest runs for many seconds under")       # and why long runs measure high
    print("     sustained load. The effect is a property of the machine, not of the")       # what it is and is not
    print("     difficulty rule, and it is the reason the hardest level gives the most")    # which figure to trust
    print("     trustworthy reading of this machine's true hash rate.")                     # and why

PARAMETER STUDY
Difficulty levels : 1,024, 4,096, 16,384, 65,536, 262,144
Trials per level  : 5
Mining now. Every number below is measured on the machine running this notebook.

  Difficulty     1,024 done:      5,775 hashes in   0.09 seconds
  Difficulty     4,096 done:     22,972 hashes in   0.34 seconds
  Difficulty    16,384 done:     68,258 hashes in   1.03 seconds
  Difficulty    65,536 done:    255,566 hashes in   3.94 seconds
  Difficulty   262,144 done:  1,409,993 hashes in  21.00 seconds

ATTEMPTS PER TRIAL
------------------------------------------------------------------------------
Difficulty   Trial 1      Trial 2      Trial 3      Trial 4      Trial 5      
------------------------------------------------------------------------------
1,024        836          3,322        311          671          635          
4,096        2,609        10,325       3,290        6,212        536          
16,384       4,316        19,341       19,059       13,639       11,903       
65,

## 4. The block reward: paying the miner when a block is found

Part A (c) asks for a simplified block reward credited to a miner address. The reward is paid by a special transaction placed first in the block, called the coinbase. It has no sender, because the units it pays did not exist before the block was found, and it pays the miner two things: a fixed subsidy created by the rules, and the fees collected from each payment in the block.

The coinbase sits inside the transaction list, and the transaction list is inside the hashed header, so the miner's address is committed to by the proof of work itself. The last part of the cell redirects the reward to another address after mining and shows the block falling apart as a result.

Sending addresses go negative here, because no balance check is performed. This lab measures mining rather than transaction validity.

In [4]:
# ============================================================================
# THE BLOCK REWARD: PAYING THE MINER WHEN A BLOCK IS FOUND
# ----------------------------------------------------------------------------
# Part A (c) asks for a simplified block reward credited to a miner address. The
# reward is paid by a special transaction placed first in the block, called the
# coinbase. It has no sender, because the coins it pays did not exist before the
# block was found, and it pays the miner two things:
#
#     subsidy : a fixed amount created by the rules, 50 units here
#     fees    : a small charge collected from each payment in the block
#
# The coinbase sits inside the transaction list, and the transaction list is inside
# the hashed header, so the miner's address is committed to by the proof of work
# itself. Changing who gets paid changes the hash, which is demonstrated at the end.
# ============================================================================

BLOCK_SUBSIDY = 50.00          # units created out of nothing and paid to whoever finds the block
FEE_PER_PAYMENT = 0.25         # a flat charge taken from each ordinary payment and paid to the miner
MINER_ADDRESS = "MINER-BK-01"  # the address credited when this miner finds a block

def make_payment(sender: str, receiver: str, amount: float) -> dict:
    """Build one ordinary payment, which pays a flat fee to whichever miner includes it."""
    return {                                  # a dictionary holding the fields of one payment
        "type": "payment",                    # marks this as an ordinary transfer rather than a reward
        "sender": sender,                     # who is paying
        "receiver": receiver,                 # who is being paid
        "amount": round(float(amount), 2),    # the value, rounded to 2 decimals so cents stay exact
        "fee": FEE_PER_PAYMENT,               # what the sender pays the miner for including this payment
    }

def make_coinbase(miner_address: str, block_index: int, total_fees: float) -> dict:
    """Build the special first transaction that pays the miner for finding this block."""
    return {                                           # the coinbase has no sender, because these coins are newly created
        "type": "coinbase",                            # marks this as the reward rather than a transfer
        "receiver": miner_address,                     # the address being paid
        "subsidy": BLOCK_SUBSIDY,                      # the fixed amount the rules create
        "fees": round(total_fees, 2),                  # everything collected from the payments in this block
        "amount": round(BLOCK_SUBSIDY + total_fees, 2),# what the miner actually receives in total
        "block": block_index,                          # naming the block stops the same coinbase being reused elsewhere
    }

def mine_block_with_reward(index: int, previous_hash: str, payments: list,
                           difficulty: int, miner_address: str) -> dict:
    """Assemble a block with its coinbase in first place, mine it, and hand back the finished block."""
    total_fees = sum(payment["fee"] for payment in payments)          # add up the fees the payments in this block will pay
    coinbase = make_coinbase(miner_address, index, total_fees)        # build the reward transaction
    transactions = [coinbase] + payments                             # the coinbase always goes first, as it does in Bitcoin

    header = build_header(                                           # assemble the header the miner will search over
        index=index,                                                 # this block's position
        previous_hash=previous_hash,                                 # the hash of the block before it
        transactions=transactions,                                   # the coinbase and the payments together
        difficulty=difficulty,                                       # the difficulty being mined at
    )
    result = mine(header, difficulty)                                # run the nonce search and measure it
    result["reward"] = coinbase["amount"]                            # remember what the miner earned from this block
    result["miner"] = miner_address                                  # and who earned it
    return result                                                    # the mining result, with the finished header inside it

def apply_block_to_balances(header: dict, balances: dict):
    """Credit the miner and move the payments, updating the running ledger of who holds what."""
    for transaction in header["transactions"]:                                   # walk every transaction in the block
        if transaction["type"] == "coinbase":                                    # the reward transaction creates new units
            balances[transaction["receiver"]] = balances.get(transaction["receiver"], 0) + transaction["amount"]  # credit the miner
        else:                                                                    # an ordinary payment moves existing units
            total_cost = transaction["amount"] + transaction["fee"]              # the sender pays the amount plus the fee
            balances[transaction["sender"]] = balances.get(transaction["sender"], 0) - total_cost      # debit the sender
            balances[transaction["receiver"]] = balances.get(transaction["receiver"], 0) + transaction["amount"]  # credit the receiver
            # the fee is not credited here, because it was already included in the coinbase amount above

# ===========================================================================
# MINE A SHORT CHAIN AND WATCH THE MINER GET PAID
# ===========================================================================
print("MINING WITH BLOCK REWARDS")                                               # heading for this block of output
print("=" * 78)                                                                  # heavy divider line
print(f"Subsidy per block : {BLOCK_SUBSIDY:.2f} units")                          # the fixed reward the rules create
print(f"Fee per payment   : {FEE_PER_PAYMENT:.2f} units")                        # what each payment contributes to the miner
print(f"Miner address     : {MINER_ADDRESS}")                                    # who gets paid in this run
print()                                                                          # blank line for readability

reward_difficulty = 16384                                # a modest difficulty, chosen so this cell runs in about a second
balances = {}                                            # the running ledger, empty before the first block is mined
previous_hash = "0" * 64                                 # the first block has no parent, so 64 zeros by convention
mined_blocks = []                                        # collects each finished block

blocks_to_mine = [                                                          # the payments each block will carry
    [make_payment("BK", "Thandi", 1500.00), make_payment("Sipho", "BK", 320.50)],   # block 1: two payments
    [make_payment("Thandi", "Lerato", 250.00)],                                      # block 2: one payment
    [make_payment("Lerato", "Naledi", 90.00), make_payment("BK", "Sipho", 40.00)],   # block 3: two payments
]

print(f"{'Block':<8}{'Nonce':<12}{'Attempts':<12}{'Seconds':<10}{'Fees':<8}{'Reward':<10}{'Hash (first 16)'}")  # table header
print("-" * 78)                                                                                                # divider

for block_number, payments in enumerate(blocks_to_mine, start=1):            # mine each block in turn
    result = mine_block_with_reward(                                         # assemble, mine and reward
        index=block_number,                                                  # the block's position in the chain
        previous_hash=previous_hash,                                         # link it to the block before
        payments=payments,                                                   # the payments it carries
        difficulty=reward_difficulty,                                        # the difficulty being mined at
        miner_address=MINER_ADDRESS,                                         # the address to credit
    )
    apply_block_to_balances(result["header"], balances)                      # update the ledger now the block is found
    mined_blocks.append(result)                                              # keep the result for the checks below
    previous_hash = result["hash"]                                           # the next block will point at this one

    fees_collected = result["header"]["transactions"][0]["fees"]             # the fee portion of the coinbase
    print(f"{block_number:<8}{result['nonce']:<12,}{result['attempts']:<12,}"  # block number, winning nonce, attempts
          f"{result['seconds']:<10.3f}{fees_collected:<8.2f}"                  # wall-clock seconds and fees collected
          f"{result['reward']:<10.2f}{result['hash'][:16]}")                   # total reward and the winning hash

print("-" * 78)                                                                # divider under the table
print()                                                                        # blank line for readability

# ---------------------------------------------------------------------------
# The resulting ledger
# ---------------------------------------------------------------------------
print("BALANCES AFTER THREE BLOCKS")                                           # heading for the ledger listing
print("-" * 78)                                                                # divider line
for address in sorted(balances):                                               # list every address in alphabetical order
    print(f"  {address:<16}{balances[address]:>12,.2f}")                       # the address and what it now holds
print("-" * 78)                                                                # divider under the listing

total_mined = sum(block["reward"] for block in mined_blocks)                    # everything the miner earned across the run
print(f"Miner earned {total_mined:,.2f} units across {len(mined_blocks)} blocks, "  # the headline earnings figure
      f"of which {BLOCK_SUBSIDY * len(mined_blocks):,.2f} is newly created subsidy.")  # and how much of it is new money
print()                                                                            # blank line for readability
print("Sending addresses go negative because no balance check is performed. This lab")  # an honest statement of the simplification
print("measures mining, not transaction validity, which Assignment 4 covered instead.")  # and where that gap is addressed
print()                                                                                  # blank line for readability

# ---------------------------------------------------------------------------
# The reward cannot be redirected after the fact
# ---------------------------------------------------------------------------
print("REDIRECTING THE REWARD AFTER MINING")                                          # heading for this test
print("-" * 78)                                                                       # divider line
stolen_header = dict(mined_blocks[0]["header"])                                       # take a copy of block 1's finished header
stolen_header["transactions"] = list(stolen_header["transactions"])                   # copy the transaction list too
stolen_header["transactions"][0] = dict(stolen_header["transactions"][0])             # and the coinbase within it
stolen_header["transactions"][0]["receiver"] = "THIEF-01"                             # THE EDIT: redirect the reward to another address

original_hash = mined_blocks[0]["hash"]                                               # the hash the block was mined with
hash_after_theft = header_hash(stolen_header)                                         # what the edited header hashes to now

print(f"  Original hash            : {original_hash}")                                # the mined hash
print(f"  Hash after redirection   : {hash_after_theft}")                             # the hash once the address changed
print(f"  Still meets the target?  : {meets_target(hash_after_theft, target_for_difficulty(reward_difficulty))}")  # almost certainly False
print()                                                                               # blank line for readability
print("Changing who gets paid changes the hash, which almost certainly no longer meets")  # why the theft fails
print("the target. The thief would have to redo the proof of work, and the moment they")  # what they would have to do instead
print("did, they would be mining their own block rather than stealing this one.")         # and why that gains them nothing

MINING WITH BLOCK REWARDS
Subsidy per block : 50.00 units
Fee per payment   : 0.25 units
Miner address     : MINER-BK-01

Block   Nonce       Attempts    Seconds   Fees    Reward    Hash (first 16)
------------------------------------------------------------------------------
1       6           7           0.000     0.50    50.50     0001a42b68b4232d
2       20,554      20,555      0.449     0.25    50.25     0003cbbdd53fc063
3       13,316      13,317      0.333     0.50    50.50     00003a34582836f4
------------------------------------------------------------------------------

BALANCES AFTER THREE BLOCKS
------------------------------------------------------------------------------
  BK                 -1,220.00
  Lerato                159.75
  MINER-BK-01           151.25
  Naledi                 90.00
  Sipho                -280.75
  Thandi              1,249.75
------------------------------------------------------------------------------
Miner earned 151.25 units across 3 block

## 5. Part B: difficulty retargeting

A network cannot fix a difficulty and leave it. Miners join and leave and hardware gets faster, so a difficulty that produced a block every ten minutes last year would produce one every ten seconds today. Bitcoin answers this by measuring how long the last 2,016 blocks actually took and adjusting the target so the next 2,016 should take the intended two weeks:

> new target = old target × (actual time ÷ expected time)

Blocks arriving too fast means the actual time was short, so the target shrinks and mining gets harder. The adjustment is clamped to a factor of four in either direction so that one freak epoch cannot send the difficulty somewhere absurd.

This cell runs the same arithmetic on a laboratory scale: epochs of five blocks and a target block time of 0.20 seconds, rather than 2,016 blocks and ten minutes. The starting difficulty is set far too low on purpose, so the algorithm has to find its way up to whatever suits the machine it is running on.

In [5]:
# ============================================================================
# PART B: DIFFICULTY RETARGETING
# ----------------------------------------------------------------------------
# A network cannot simply fix a difficulty and leave it. Miners join and leave, and
# hardware gets faster, so a difficulty that produced a block every ten minutes last
# year would produce one every ten seconds today. Bitcoin answers this by measuring
# how long the last 2,016 blocks actually took and adjusting the target so that the
# next 2,016 should take the intended two weeks:
#
#       new target = old target  x  (actual time / expected time)
#
# Blocks arriving too quickly means the actual time was short, so the target shrinks
# and mining gets harder. The adjustment is clamped to a factor of four in either
# direction, so that one freak epoch cannot send the difficulty somewhere absurd.
#
# This cell uses the same arithmetic on a laboratory scale: epochs of 5 blocks and a
# target block time of 0.20 seconds rather than 2,016 blocks and ten minutes.
# ============================================================================

EPOCH_BLOCKS = 5                 # how many blocks are mined before the difficulty is reconsidered
TARGET_BLOCK_SECONDS = 0.20      # how long each block is meant to take on this machine
MAX_ADJUSTMENT_FACTOR = 4        # the difficulty may not move by more than this in one step, as in Bitcoin

def retarget(old_difficulty: int, actual_seconds: float, expected_seconds: float) -> int:
    """Work out the difficulty for the next epoch from how long the last one actually took."""
    ratio = expected_seconds / actual_seconds        # above 1 means blocks came too fast, so difficulty must rise
    if ratio > MAX_ADJUSTMENT_FACTOR:                # refuse to raise difficulty by more than the clamp allows
        ratio = MAX_ADJUSTMENT_FACTOR                # cap the increase
    if ratio < 1 / MAX_ADJUSTMENT_FACTOR:            # refuse to lower it by more than the clamp allows either
        ratio = 1 / MAX_ADJUSTMENT_FACTOR            # cap the decrease
    new_difficulty = int(old_difficulty * ratio)     # apply the adjustment, rounding down to a whole number
    return max(1, new_difficulty)                    # never allow difficulty to fall below 1, which would be no work at all

def mine_epoch(start_index: int, difficulty: int, miner_address: str) -> dict:
    """Mine one epoch of blocks at a fixed difficulty and measure how long the whole epoch took."""
    attempts_total = 0                                       # every hash computed during this epoch
    epoch_start = time.perf_counter()                        # the clock reading before the first block
    for offset in range(EPOCH_BLOCKS):                       # mine the blocks of this epoch one after another
        block_header = build_header(                         # each block gets a header of its own
            index=start_index + offset,                      # varying the index keeps every search independent
            previous_hash="0" * 64,                          # a fixed placeholder parent, since this study is about timing
            transactions=[make_coinbase(miner_address, start_index + offset, 0.0)],  # just the reward, no payments
            difficulty=difficulty,                           # the difficulty this epoch is being mined at
        )
        block_result = mine(block_header, difficulty)        # mine it
        attempts_total += block_result["attempts"]           # add its hashes to the epoch total
    epoch_seconds = time.perf_counter() - epoch_start        # wall-clock time for the whole epoch
    return {                                                              # what the retargeting decision needs
        "attempts": attempts_total,                                       # hashes computed across the epoch
        "seconds": epoch_seconds,                                         # wall-clock seconds the epoch took
        "seconds_per_block": epoch_seconds / EPOCH_BLOCKS,                # the figure being steered towards the target
        "hash_rate": attempts_total / epoch_seconds,                      # the machine's measured speed this epoch
    }

# ===========================================================================
# STARTING FROM A DELIBERATELY WRONG DIFFICULTY AND LETTING IT CORRECT ITSELF
# ===========================================================================
print("DIFFICULTY RETARGETING")                                                  # heading for this block of output
print("=" * 78)                                                                  # heavy divider line
print(f"Epoch length       : {EPOCH_BLOCKS} blocks")                             # how often the difficulty is reconsidered
print(f"Target block time  : {TARGET_BLOCK_SECONDS:.2f} seconds")                # the block time being steered towards
print(f"Adjustment clamp   : {MAX_ADJUSTMENT_FACTOR}x in either direction")      # the limit on one adjustment
print()                                                                          # blank line for readability
print("Starting difficulty is set far too low on purpose, so blocks arrive much")  # what the demonstration sets up
print("faster than intended and the algorithm has to correct upwards.")            # and what it will have to do about it
print()                                                                            # blank line for readability

current_difficulty = 256                                  # deliberately far too easy for a modern processor
expected_epoch_seconds = EPOCH_BLOCKS * TARGET_BLOCK_SECONDS   # how long a well-tuned epoch should take
epoch_history = []                                        # collects one record per epoch for the table below

print(f"{'Epoch':<8}{'Difficulty':<14}{'Secs/block':<14}{'Target':<10}{'Ratio':<10}{'Next difficulty'}")  # table header
print("-" * 78)                                                                                          # divider

for epoch_number in range(1, 9):                                        # run eight epochs, which is enough to converge
    epoch = mine_epoch(                                                 # mine one epoch at the current difficulty
        start_index=epoch_number * 100,                                 # spacing the indexes apart keeps searches independent
        difficulty=current_difficulty,                                  # the difficulty this epoch uses
        miner_address=MINER_ADDRESS,                                    # the address credited in each coinbase
    )
    next_difficulty = retarget(                                         # decide the difficulty for the next epoch
        old_difficulty=current_difficulty,                              # what was just used
        actual_seconds=epoch["seconds"],                                # how long the epoch really took
        expected_seconds=expected_epoch_seconds,                        # how long it should have taken
    )
    ratio = expected_epoch_seconds / epoch["seconds"]                   # the raw adjustment before the clamp is applied

    epoch_history.append({                                              # keep the numbers for the discussion below
        "epoch": epoch_number,                                          # which epoch this was
        "difficulty": current_difficulty,                               # the difficulty it used
        "seconds_per_block": epoch["seconds_per_block"],                # the measured block time
        "hash_rate": epoch["hash_rate"],                                # the measured hash rate
    })

    print(f"{epoch_number:<8}{current_difficulty:<14,}"                 # epoch number and the difficulty it used
          f"{epoch['seconds_per_block']:<14.4f}{TARGET_BLOCK_SECONDS:<10.2f}"  # measured block time against the target
          f"{ratio:<10.2f}{next_difficulty:,}")                          # the raw ratio and the difficulty chosen next

    current_difficulty = next_difficulty                                # carry the new difficulty into the next epoch

print("-" * 78)                                                          # divider under the table
print()                                                                  # blank line for readability

# ---------------------------------------------------------------------------
# What the algorithm was aiming at
# ---------------------------------------------------------------------------
measured_hash_rate = statistics.mean(e["hash_rate"] for e in epoch_history)      # average speed observed across all epochs
ideal_difficulty = measured_hash_rate * TARGET_BLOCK_SECONDS                     # the difficulty that suits this machine exactly

print("WHERE IT WAS HEADING")                                                            # heading for the interpretation
print(f"  Mean hash rate across the run : {measured_hash_rate:,.0f} hashes per second")  # the machine's measured speed
print(f"  Difficulty that fits it       : {ideal_difficulty:,.0f}")                      # hash rate multiplied by the target block time
print(f"  Difficulty after eight epochs : {current_difficulty:,}")                       # where the algorithm actually arrived
print()                                                                                  # blank line for readability

print("READING THE TABLE")                                                                # heading for the discussion
print("  The first epochs show the clamp doing its job. The raw ratio demanded a jump of") # what the early rows show
print("  far more than fourfold, and the algorithm refused, climbing in steps of four")    # and how the clamp responded
print("  instead. This is why Bitcoin reaches a correct difficulty over several epochs")   # the consequence in a real network
print("  rather than one, and it is deliberate: a single freak epoch, or a timestamp a")   # why the clamp is wanted
print("  miner lied about, cannot move the difficulty somewhere absurd in one step.")      # the attack the clamp blunts
print()                                                                                   # blank line for readability
print("  Once near the right difficulty the figure oscillates rather than settling, which") # the behaviour after convergence
print("  is the small sample problem again: five blocks is far too few to measure a block") # why it oscillates
print("  time reliably, so the algorithm partly chases noise. Bitcoin averages over 2,016") # what Bitcoin does instead
print("  blocks for exactly this reason, roughly two weeks of evidence per decision.")      # and how much evidence that buys

DIFFICULTY RETARGETING
Epoch length       : 5 blocks
Target block time  : 0.20 seconds
Adjustment clamp   : 4x in either direction

Starting difficulty is set far too low on purpose, so blocks arrive much
faster than intended and the algorithm has to correct upwards.

Epoch   Difficulty    Secs/block    Target    Ratio     Next difficulty
------------------------------------------------------------------------------
1       256           0.0034        0.20      58.89     1,024
2       1,024         0.0128        0.20      15.62     4,096
3       4,096         0.0562        0.20      3.56      14,572
4       14,572        0.0686        0.20      2.92      42,489
5       42,489        0.8180        0.20      0.24      10,622
6       10,622        0.0877        0.20      2.28      24,234
7       24,234        0.3810        0.20      0.52      12,721
8       12,721        0.2327        0.20      0.86      10,933
------------------------------------------------------------------------------

## 6. Part B continued: what happens when the hash rate changes

The previous section corrected a difficulty that was wrong to begin with. The harder case is a difficulty that was right and then stopped being right, because miners left or hardware changed. That is the situation retargeting exists for.

The slower miner is simulated honestly rather than faked. `mine_at_rate()` performs extra throwaway hashes on every attempt, each costing the same as a real attempt, so the processor genuinely spends longer per attempt and the measured hash rate genuinely falls. No timing figure is invented. The handicap switches on at epoch 4, and the difficulty has to find its way back down.

The cell closes with the explanation Part B asks for: why the rest of this lab holds difficulty fixed.

In [6]:
# ============================================================================
# PART B CONTINUED: WHAT HAPPENS WHEN THE HASH RATE CHANGES
# ----------------------------------------------------------------------------
# The previous cell corrected a difficulty that was wrong to begin with. The harder
# case is a difficulty that was right and then stopped being right, because miners
# left the network or hardware changed. That is the situation retargeting exists for.
#
# A slower miner is simulated honestly here rather than faked: mine_at_rate() performs
# a number of extra throwaway hashes on every attempt, so the processor really does
# spend longer per attempt and the measured hash rate really does fall. Nothing in the
# timing is invented. Halfway through the run the extra work switches on, cutting the
# effective rate roughly in half, and the difficulty has to find its way back down.
# ============================================================================

def mine_at_rate(header: dict, difficulty: int, extra_work: int = 0) -> dict:
    """Mine a block while optionally doing extra throwaway work per attempt, to imitate a slower miner."""
    target = target_for_difficulty(difficulty)       # the ceiling a winning hash must fall below
    working_header = dict(header)                    # copy the header so the caller's dictionary is untouched
    working_header["difficulty"] = difficulty        # commit to the difficulty actually being mined
    attempts = 0                                     # counts only real attempts, not the throwaway work
    nonce = 0                                        # the search begins at zero as always

    start_time = time.perf_counter()                 # clock reading before the search starts

    while True:                                                  # keep guessing until a winning hash appears
        working_header["nonce"] = nonce                          # write the current guess into the header
        candidate_hash = header_hash(working_header)             # fingerprint the header with that guess
        attempts += 1                                            # that was one real attempt
        for _ in range(extra_work):                              # burn processor time to imitate slower hardware
            header_hash(working_header)                          # one unit of extra work costs the same as one real attempt,
            # so extra_work=1 roughly halves the effective rate. The result is discarded: only the time spent matters.
        if meets_target(candidate_hash, target):                 # does the hash fall at or below the target?
            break                                                # yes, the block is mined
        nonce += 1                                               # no, try the next nonce

    elapsed_seconds = time.perf_counter() - start_time           # wall-clock time the search really took
    return {                                                     # the same shape of result as mine()
        "nonce": nonce,                                          # the winning nonce
        "hash": candidate_hash,                                  # the winning hash
        "attempts": attempts,                                    # real attempts, excluding throwaway work
        "seconds": elapsed_seconds,                              # measured wall-clock seconds
    }

def mine_epoch_at_rate(start_index: int, difficulty: int, extra_work: int) -> dict:
    """Mine one epoch at a given difficulty and a given handicap, and measure the whole epoch."""
    attempts_total = 0                                           # every real attempt made during this epoch
    epoch_start = time.perf_counter()                            # clock reading before the first block
    for offset in range(EPOCH_BLOCKS):                           # mine the blocks of the epoch in turn
        block_header = build_header(                             # each block gets its own header
            index=start_index + offset,                          # varying the index keeps the searches independent
            previous_hash="0" * 64,                              # a fixed placeholder parent, since this study is about timing
            transactions=[make_coinbase(MINER_ADDRESS, start_index + offset, 0.0)],  # just the reward, no payments
            difficulty=difficulty,                               # the difficulty for this epoch
        )
        block_result = mine_at_rate(block_header, difficulty, extra_work)  # mine it with the handicap applied
        attempts_total += block_result["attempts"]               # add its attempts to the epoch total
    epoch_seconds = time.perf_counter() - epoch_start            # wall-clock time for the whole epoch
    return {                                                                  # what the retargeting decision needs
        "attempts": attempts_total,                                           # real attempts across the epoch
        "seconds": epoch_seconds,                                             # wall-clock seconds the epoch took
        "seconds_per_block": epoch_seconds / EPOCH_BLOCKS,                    # the figure being steered to the target
        "hash_rate": attempts_total / epoch_seconds,                          # the effective rate, handicap included
    }

# ===========================================================================
# HALF THE MINING POWER DISAPPEARS PART WAY THROUGH
# ===========================================================================
print("RETARGETING AFTER A CHANGE IN HASH RATE")                                 # heading for this block of output
print("=" * 78)                                                                  # heavy divider line
print("Epochs 1 to 3 run at full speed. From epoch 4 onwards each attempt carries")  # what the demonstration does
print("extra throwaway work, so the effective hash rate falls by roughly half.")      # and the effect that has
print()                                                                              # blank line for readability

adaptive_difficulty = max(1024, int(ideal_difficulty))     # start near the correct difficulty measured in the previous cell
adaptation_history = []                                    # collects one record per epoch

print(f"{'Epoch':<8}{'Speed':<12}{'Difficulty':<14}{'Secs/block':<14}{'Hashes/sec':<14}{'Next difficulty'}")  # table header
print("-" * 78)                                                                                              # divider

for epoch_number in range(1, 9):                                        # run eight epochs in total
    extra_work = 0 if epoch_number <= 3 else 1                          # the handicap switches on at epoch 4
    speed_label = "full" if extra_work == 0 else "halved"               # a readable label for the table

    epoch = mine_epoch_at_rate(                                         # mine the epoch at this difficulty and speed
        start_index=1000 + epoch_number * 100,                          # spaced indexes keep every search independent
        difficulty=adaptive_difficulty,                                 # the difficulty currently in force
        extra_work=extra_work,                                          # the handicap, zero or one
    )
    next_difficulty = retarget(                                         # decide the difficulty for the next epoch
        old_difficulty=adaptive_difficulty,                             # what was just used
        actual_seconds=epoch["seconds"],                                # how long the epoch really took
        expected_seconds=expected_epoch_seconds,                        # how long it should have taken
    )

    adaptation_history.append({                                         # keep the numbers for the summary below
        "epoch": epoch_number,                                          # which epoch this was
        "extra_work": extra_work,                                       # whether the handicap was on
        "seconds_per_block": epoch["seconds_per_block"],                # the measured block time
        "hash_rate": epoch["hash_rate"],                                # the measured effective rate
    })

    print(f"{epoch_number:<8}{speed_label:<12}{adaptive_difficulty:<14,}"    # epoch, speed label and difficulty used
          f"{epoch['seconds_per_block']:<14.4f}{epoch['hash_rate']:<14,.0f}"  # measured block time and effective rate
          f"{next_difficulty:,}")                                             # the difficulty chosen for next time

    adaptive_difficulty = next_difficulty                               # carry the new difficulty forward

print("-" * 78)                                                          # divider under the table
print()                                                                  # blank line for readability

full_speed_rate = statistics.mean(e["hash_rate"] for e in adaptation_history if e["extra_work"] == 0)   # rate before the change
slowed_rate = statistics.mean(e["hash_rate"] for e in adaptation_history if e["extra_work"] == 1)       # rate after the change
full_speed_block = statistics.mean(e["seconds_per_block"] for e in adaptation_history if e["extra_work"] == 0)  # block time before
slowed_block = statistics.mean(e["seconds_per_block"] for e in adaptation_history if e["extra_work"] == 1)      # block time after

print("EFFECT OF THE CHANGE")                                                            # heading for the summary
print(f"  Hash rate at full speed  : {full_speed_rate:,.0f} per second")                 # measured rate before the handicap
print(f"  Hash rate when slowed    : {slowed_rate:,.0f} per second")                     # measured rate after it
print(f"  Effective reduction      : {(1 - slowed_rate / full_speed_rate) * 100:.0f} percent")  # how much power was removed
print(f"  Mean block time before   : {full_speed_block:.4f} seconds")                    # how long blocks took before
print(f"  Mean block time after    : {slowed_block:.4f} seconds")                        # and after, once difficulty responded
print(f"  Target block time        : {TARGET_BLOCK_SECONDS:.4f} seconds")                # the figure both are steered towards
print()                                                                                  # blank line for readability
print("Losing half the mining power does not halve the security of the chain, and it")   # what the table demonstrates
print("does not permanently slow it either. Blocks are slower for one epoch, then the")  # the transitional cost
print("difficulty falls and the block time returns towards its target. The chain simply") # and what happens after
print("becomes cheaper to attack, because less work now stands behind each block.")       # the security consequence
print()                                                                                   # blank line for readability

# ---------------------------------------------------------------------------
# Why the rest of this lab fixes the difficulty
# ---------------------------------------------------------------------------
print("WHY THE REST OF THIS LAB USES A FIXED DIFFICULTY")                                  # heading for the required explanation
print("=" * 78)                                                                            # heavy divider line
print("  1. Reproducibility. With a fixed difficulty and fixed timestamps, every nonce")   # the first reason
print("     and hash in this notebook is identical on any machine. Retargeting makes the")  # what retargeting would break
print("     difficulty a function of local processor speed, so nobody else could")          # because the result depends on hardware
print("     reproduce the values, only the shape of the behaviour.")                        # leaving only the pattern reproducible
print()                                                                                     # blank line for readability
print("  2. One variable at a time. Part A (b) measures how attempts and wall-clock time")  # the second reason
print("     respond to difficulty. If the difficulty moved during the run, nothing could")  # the confound that would create
print("     be attributed to anything, which is why the parameter study holds it still.")   # and why the study fixes it
print()                                                                                     # blank line for readability
print("  3. There is no network here, and no honest clock. Bitcoin retargets on timestamps")  # the third reason
print("     reported by miners who may lie, which is why the clamp and the median-of-eleven")  # the defences Bitcoin needs
print("     timestamp rule exist. A single process with fixed timestamps has none of that")    # what a lab lacks
print("     context, so retargeting here demonstrates the arithmetic rather than the")         # what this cell actually shows
print("     adversarial problem it was designed to solve.")                                    # and what it cannot show
print()                                                                                        # blank line for readability
print("  4. Epochs of five blocks are too short to measure anything reliably, as the")         # the fourth reason
print("     oscillation in the previous cell showed. Fixing the difficulty avoids reporting")  # what would otherwise happen
print("     noise as though it were a result.")                                                # and the risk that creates

RETARGETING AFTER A CHANGE IN HASH RATE
Epochs 1 to 3 run at full speed. From epoch 4 onwards each attempt carries
extra throwaway work, so the effective hash rate falls by roughly half.

Epoch   Speed       Difficulty    Secs/block    Hashes/sec    Next difficulty
------------------------------------------------------------------------------
1       full        11,312        0.1478        52,913        15,310
2       full        15,310        0.2899        55,316        10,562
3       full        10,562        0.1465        55,995        14,419
4       halved      14,419        0.2047        28,842        14,091
5       halved      14,091        0.3781        28,086        7,452
6       halved      7,452         0.3863        29,447        3,858
7       halved      3,858         0.1211        29,349        6,374
8       halved      6,374         0.2002        28,821        6,368
------------------------------------------------------------------------------

EFFECT OF THE CHANGE
  Hash